# D2-04 · Modern detectors in practice: YOLO, DETR, fine-tuning, deployment

### Steps
1. The YOLO family in one page, and where today's models sit on the speed / accuracy curve
2. DETR: set prediction with Hungarian matching - no anchors, no NMS
3. Open-vocabulary detection: classes as text prompts
4. Fine-tune a modern anchor-free detector (FCOS) and compare it with D2-02's Faster R-CNN
5. Deployment: input size vs latency, ONNX export, quantisation, monitoring
6. Which detector would you pick? Three scenarios

### Topics covered and learning objectives
- Name what each YOLO generation changed and why NMS-free training matters
- Implement DETR's matching cost and the Hungarian assignment
- Adapt a pretrained detector to a new dataset and evaluate it honestly
- Measure latency and reason about the resolution / accuracy / speed trade-off
- Choose a detector for a real constraint (latency, data, licence, open vocabulary)

### Time estimate
~75 min in class (guided + hands-on)

### Prerequisites
D2-01 to D2-03.

## 0 · Setup

<details class="api-ref" data-api="np.set_printoptions,torch.manual_seed,torchkit.get_device"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.set_printoptions</code> · <code>torch.manual_seed</code> · <code>torchkit.get_device</code></summary>
<ul><li><code>np.set_printoptions(precision=2, suppress=True)</code> &nbsp; Changes how NumPy prints arrays for the rest of the session: precision=2 shows two decimals, suppress=True writes tiny numbers as 0. instead of in scientific notation (2e-05). Only the display changes, never the values.</li><li><code>torch.manual_seed(seed)</code> &nbsp; Fixes PyTorch&#x27;s random number generator, so random weights, shuffles and augmentations come out the same on every run. The notebooks call it in the setup cell and again right before creating a model, so that its initial weights are repeatable.</li><li><code>device = torchkit.get_device()</code> &nbsp; Course helper. Returns torch.device(&quot;cuda&quot;) when PyTorch can use an NVIDIA GPU, torch.device(&quot;mps&quot;) on an Apple-silicon Mac, otherwise torch.device(&quot;cpu&quot;), and prints which one it picked (for the CPU, with the number of threads PyTorch uses). Models and batches are then moved there with .to(device).</li></ul></details>

In [ ]:
import sys, pathlib, time
for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:   # find the cvcourse package
    if (p / "cvcourse").is_dir():
        sys.path.insert(0, str(p)); break

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from cvcourse import viz, boxes, shapes, tests, torchkit, day2, labs

%matplotlib inline
plt.rcParams["figure.dpi"] = 90
np.set_printoptions(precision=3, suppress=True)
torch.manual_seed(0)
device = torchkit.get_device()
GREEN, RED, GREY, BLUE, ORANGE, PURPLE = "#54A24B", "#E45756", "#9a9a9a", "#4C78A8", "#F58518", "#B279A2"

## 1 · The YOLO family in one page

"YOLO" is no longer one model but a brand shared by several groups. What each step changed:

| version | year | what changed |
|---|---|---|
| YOLOv1 | 2016 | one grid, 2 boxes per cell, one forward pass (our D2-03 mini detector) |
| YOLOv2 / 9000 | 2017 | anchors from k-means on the dataset, BatchNorm, multi-scale training |
| YOLOv3 | 2018 | 3 scales (FPN-like), Darknet-53, independent sigmoid classes |
| YOLOv4 / v5 | 2020 | CSP backbones, Mosaic augmentation (four training images tiled into one), CIoU loss; v5 = PyTorch + Ultralytics tooling |
| YOLOX | 2021 | **anchor-free**, **decoupled** cls / box heads, SimOTA dynamic assignment |
| YOLOv6 / v7 | 2022 | re-parameterised blocks (train multi-branch, deploy as one conv), better assignment |
| YOLOv8 | 2023 | anchor-free decoupled head with **Distribution Focal Loss**, task-aligned assignment |
| YOLOv9 | 2024 | programmable gradient information (auxiliary reversible branch) |
| YOLOv10 | 2024 | **NMS-free**: one-to-many + one-to-one heads trained together, only the one-to-one head at inference |
| YOLO11 / 12 | 2024-25 | efficiency tweaks; v12 brings attention blocks into the backbone |
| YOLO26 | 2025-26 | end-to-end NMS-free by default, DFL removed for simpler export to edge devices |

The arc: anchors appeared (v2), multi-scale arrived (v3), anchors disappeared again (YOLOX, v8),
and the last hand-written step - NMS - is being trained away (v10, 26), following DETR.
Meanwhile transformer detectors became real-time (RT-DETR, 2023).

In [ ]:
# Speed / accuracy of current real-time detectors (published numbers: COCO val mAP@[.5:.95],
# latency per image on an NVIDIA T4 GPU with TensorRT FP16, YOLO numbers without NMS; approximate - read the trend)
families = {
    "YOLOv8 (n-x)":      ([1.47, 2.66, 5.86, 9.06, 14.37], [37.3, 44.9, 50.2, 52.9, 53.9], BLUE),
    "YOLOv10 (n-x)":     ([1.84, 2.49, 4.74, 5.74, 7.28, 10.70], [38.5, 46.3, 51.1, 52.5, 53.2, 54.4], ORANGE),
    "YOLO11 (n-x)":      ([1.5, 2.5, 4.7, 6.2, 11.3], [39.5, 47.0, 51.5, 53.4, 54.7], GREEN),
    "RT-DETR (R18-R101)": ([4.6, 9.2, 13.5], [46.5, 53.1, 54.3], PURPLE),
}
fig, ax = plt.subplots(figsize=(10, 5))
for name, (lat, ap, c) in families.items():
    ax.plot(lat, ap, "o-", color=c, lw=2, label=name)
for name, ap in [("torchvision Faster R-CNN R50-FPN v2", 46.7), ("torchvision FCOS R50-FPN", 39.2), ("torchvision RetinaNet R50-FPN", 36.4)]:
    ax.axhline(ap, color=GREY, ls=":", lw=1); ax.text(24.8, ap + 0.15, name, fontsize=8, ha="right", va="bottom", color="#555")
ax.set_xlabel("latency (ms / image, T4 TensorRT FP16)"); ax.set_ylabel("COCO mAP@[.5:.95]"); ax.set_xlim(0, 25)
ax.grid(alpha=.3); ax.legend(loc="lower right"); ax.set_title("Real-time detectors 2023-2025: each curve = one family, small to large")
plt.tight_layout(); plt.show()

The torchvision models we use in class (dotted lines, accuracy only - their latency was never
optimised for deployment) are **teaching models**: readable, BSD-licensed, easy to modify. For
production on a GPU you would pick a curve on the left part of this plot. Note the licences:
Ultralytics YOLO (v5, v8, 11, 26) is **AGPL-3.0** (or a paid licence), RT-DETR (Apache-2.0) and
torchvision (BSD) are permissive - a real constraint at work.

## 2 · DETR: detection as set prediction

DETR (Carion et al. 2020) removes every hand-designed component of the previous notebooks:

1. a CNN backbone produces a feature map; it is flattened into a sequence of tokens (+ positional encodings:
   a vector added to every token that says where in the image it came from, because attention itself ignores order);
2. a **transformer encoder** lets every location attend to every other one (global context);
3. a **decoder** takes $N = 100$ learned **object queries**; each query attends to the image and outputs one
   prediction: class probabilities over $C + 1$ classes (incl. "no object") and a box (normalised cxcywh);
4. training: **Hungarian matching** finds the one-to-one assignment between the $N$ predictions and the
   $G$ ground-truth objects with the lowest total cost; matched queries learn their object, all others
   learn "no object".

Because each object is assigned to exactly one query, the network learns **not to produce duplicates**:
no anchors, no NMS, no IoU thresholds.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 3.6)); ax.axis("off"); ax.set_xlim(0, 15); ax.set_ylim(0, 3.6)
items = [(0.1, "image", "#dddddd"), (1.9, "CNN\nbackbone", "#9ecae1"), (3.7, "flatten +\npositional enc.\n(HW tokens)", "#c6dbef"),
         (5.7, "transformer\nencoder", ORANGE), (7.8, "transformer\ndecoder", "#fdd0a2"), (10.1, "N predictions\n(class, box)", GREEN),
         (12.4, "Hungarian\nmatching\n(train only)", PURPLE)]
for x, t, c in items:
    ax.add_patch(FancyBboxPatch((x, 1.1), 1.6, 1.5, boxstyle="round,pad=0.05", fc=c, ec="k", alpha=0.85))
    ax.text(x + 0.8, 1.85, t, ha="center", va="center", fontsize=9)
for x0, x1 in [(1.75, 1.9), (3.55, 3.7), (5.35, 5.7), (7.35, 7.8), (9.45, 10.1), (11.75, 12.4)]:
    ax.annotate("", (x1, 1.85), (x0, 1.85), arrowprops=dict(arrowstyle="-|>", lw=1.6))
ax.text(8.6, 3.25, "N learned object queries", ha="center", fontsize=9)
ax.annotate("", (8.6, 2.65), (8.6, 3.1), arrowprops=dict(arrowstyle="-|>", lw=1.6))
ax.text(13.2, 0.6, "vs ground-truth set", ha="center", fontsize=9)
ax.text(10.9, 0.6, "no NMS at inference", ha="center", fontsize=9, color=GREEN, weight="bold")
plt.show()

### The matching cost

For prediction $i$ and ground truth $j$ DETR uses

$$\mathcal{C}_{ij} = -\lambda_{\text{cls}}\,\hat p_i(c_j) + \lambda_{L1}\,\lVert \hat b_i - b_j \rVert_1 - \lambda_{\text{giou}}\,\mathrm{GIoU}(\hat b_i, b_j),
\qquad \lambda_{\text{cls}} = 1,\ \lambda_{L1} = 5,\ \lambda_{\text{giou}} = 2,$$

where $\hat p_i(c_j)$ is the probability that prediction $i$ gives to the class $c_j$ of ground truth $j$,
and boxes are normalised $(c_x, c_y, w, h)$. The **Hungarian algorithm** (Kuhn 1955) finds the
assignment $\sigma$ (ground truth $j$ gets query $\sigma(j)$, no query used twice) that minimises
$\sum_j \mathcal{C}_{\sigma(j) j}$ exactly in $O(N^3)$. `scipy.optimize.linear_sum_assignment` implements it
and accepts rectangular matrices (more queries than objects).

### Exercise 1 · `hungarian_match`

Return `(query_idx, gt_idx)` for a cost matrix `(Q, G)`, numpy or torch: two integer arrays of
length `G` (one pair per ground-truth object), so that `cost[query_idx, gt_idx].sum()` is minimal.

<details class="api-ref" data-api="scipy.optimize.linear_sum_assignment,torch.is_tensor"><summary><b>Functions you will need</b> (click to open): <code>scipy.optimize.linear_sum_assignment</code> · <code>torch.is_tensor</code></summary>
<p><b><code>rows, cols = linear_sum_assignment(cost)</code></b><br>The Hungarian algorithm. Given a cost matrix (Q, G), it picks min(Q, G) pairs, using each row and each column at most once, with the smallest total cost, and returns their row and column indices. With more rows (queries) than columns (objects), every object gets exactly one query and the rest stay unmatched.</p><pre>&gt;&gt;&gt; from scipy.optimize import linear_sum_assignment
&gt;&gt;&gt; cost = np.array([[4., 1.],             # 3 queries (rows) x 2 objects (columns)
...                  [2., 3.],
...                  [5., 6.]])
&gt;&gt;&gt; rows, cols = linear_sum_assignment(cost)
&gt;&gt;&gt; rows, cols                             # query 0 -&gt; object 1, query 1 -&gt; object 0
(array([0, 1]), array([1, 0]))
&gt;&gt;&gt; float(cost[rows, cols].sum())          # 1 + 2; query 2 is left over
3.0</pre><p><i>Watch out:</i> It works on NumPy arrays: convert a cost computed by a model with .detach().cpu().numpy() first. maximize=True finds the largest total instead (for scores rather than costs).</p><p>See also: <code>torch.cdist</code>, <code>torchvision.ops.generalized_box_iou</code></p><p><b><code>torch.is_tensor(obj)</code></b><br>True if obj is a PyTorch tensor (also an nn.Parameter or a torchvision image), False for NumPy arrays, lists and numbers. Lets a helper accept both: tensors are converted with .detach().cpu().numpy(), everything else with np.asarray.</p><pre>&gt;&gt;&gt; torch.is_tensor(torch.zeros(2)), torch.is_tensor(np.zeros(2)), torch.is_tensor([1, 2])
(True, False, False)</pre><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="Tensor.detach,Tensor.cpu,Tensor.numpy,np.asarray"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>Tensor.detach</code> · <code>Tensor.cpu</code> · <code>Tensor.numpy</code> · <code>np.asarray</code></summary>
<ul><li><code>x.detach()</code> &nbsp; Returns the same data as a tensor that autograd no longer tracks (requires_grad=False), without copying. Needed to turn a weight or a model output into NumPy, or to keep it without keeping its whole computation alive.</li><li><code>x.cpu()</code> &nbsp; Returns a copy of the tensor in main memory (the CPU), or the tensor itself if it is already there. Needed before .numpy() or plotting when the result was computed on a GPU: x.detach().cpu().numpy().</li><li><code>x.numpy()</code> &nbsp; Returns the tensor&#x27;s data as a NumPy array without copying: both share the same memory, so changing one changes the other. Used to hand results to Matplotlib or to NumPy code.</li><li><code>np.asarray(obj)</code> &nbsp; Converts its argument (a list, a PIL image, a CPU tensor, ...) to a NumPy array; an array is returned as it is, without a copy. A grey PIL image becomes a (H, W) uint8 array, a colour one (H, W, 3).</li></ul></details>

In [ ]:
from scipy.optimize import linear_sum_assignment

def hungarian_match(cost):
    '''cost (Q, G), a NumPy array or a torch tensor -> (query_idx, gt_idx): two integer arrays of length G,
    one (query, ground truth) pair per object, so that cost[query_idx, gt_idx].sum() is as small as possible.'''
    # 1. SciPy works on NumPy arrays: a tensor is detached from autograd, moved to the CPU and converted
    if torch.is_tensor(cost):
        cost = cost.detach()
        cost = cost.cpu()
        cost = cost.numpy()
    else:
        cost = np.asarray(cost)
    # 2. the Hungarian algorithm: the cheapest one-to-one assignment
    query_idx, gt_idx = linear_sum_assignment(cost)      # (G,) each, when there are more queries than objects
    # 3. both index arrays
    return query_idx, gt_idx

tests.check_hungarian(hungarian_match)

<details class="api-ref" data-api="Tensor.argmin,np.delete"><summary><b>Functions used here</b> (click to open): <code>Tensor.argmin</code> · <code>np.delete</code></summary>
<p><b><code>x.argmin(dim=None)</code></b><br>Index of the smallest value: of the flattened tensor without dim, otherwise one index per row or column. NumPy arrays have the same method (axis=); D2-04 uses it on one column of a cost matrix to find the cheapest query for an object.</p><pre>&gt;&gt;&gt; cost = torch.tensor([[4., 1.], [2., 3.], [5., 0.5]])
&gt;&gt;&gt; cost[:, 0].argmin()              # cheapest query for object 0
tensor(1)
&gt;&gt;&gt; cost.argmin(dim=0)               # for every object (column)
tensor([1, 2])
&gt;&gt;&gt; cost.argmin()                    # no dim: an index into the flattened tensor
tensor(5)</pre><p>See also: <code>Tensor.argmax</code>, <code>Tensor.min</code></p><p><b><code>np.delete(a, index, axis=None)</code></b><br>Returns a copy of a without the given element(s); with axis, whole rows (axis=0) or columns (axis=1) are removed. The original array is not changed.</p><pre>&gt;&gt;&gt; np.delete(np.arange(5), 2)
array([0, 1, 3, 4])
&gt;&gt;&gt; np.delete(np.array([[1, 2], [3, 4], [5, 6]]), 0, axis=0)      # drop row 0
array([[3, 4],
       [5, 6]])</pre><p><i>Watch out:</i> Without axis the array is flattened first: np.delete(m, 0) on a 2-D m returns a 1-D array.</p><p>See also: <code>np.where</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.array,np.arange,Tensor.sum"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.array</code> · <code>np.arange</code> · <code>Tensor.sum</code></summary>
<ul><li><code>np.array(data, dtype=None)</code> &nbsp; Creates a NumPy array from a (nested) list, copying the data. The dtype is inferred: integers give int64, any float gives float64; dtype=float gives floats from integer literals.</li><li><code>np.arange(stop) / np.arange(start, stop, step, dtype=None)</code> &nbsp; Like Python&#x27;s range, as a NumPy array: from start (default 0) up to but not including stop. With reshape, the quickest way to build a small test array with distinct values.</li><li><code>x.sum(dim=None, keepdim=False)</code> &nbsp; Adds up all elements (giving a 0-dimensional tensor), or only along dim, which then disappears from the shape: for a (2, 3) tensor, sum(dim=0) adds the two rows together and gives one value per column, shape (3,). dim can be a tuple; keepdim=True keeps the summed dimension with size 1.</li></ul></details>

In [ ]:
# The cartoon: 3 queries, 2 objects. Greedy "each GT takes its cheapest query" vs the optimal matching
cost = np.array([[0.20, 0.30],
                 [0.25, 0.90],
                 [0.80, 0.85]])
r, c = day2.hungarian_match(cost)
# greedy: GT A takes its cheapest query, then GT B takes the cheapest of the queries that are left
query_for_a = int(cost[:, 0].argmin())                         # query 1 (index 0), cost 0.20
queries_left = np.delete(np.arange(3), query_for_a)            # indices [1, 2]
query_for_b = int(queries_left[cost[queries_left, 1].argmin()])
greedy = [query_for_a, query_for_b]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), gridspec_kw=dict(width_ratios=[1, 1, 1.3]))
for ax, assign, title in [(axes[0], list(zip(greedy, [0, 1])), f"greedy (GT A first): total {cost[greedy[0], 0] + cost[greedy[1], 1]:.2f}"),
                          (axes[1], list(zip(r, c)), f"Hungarian (optimal): total {cost[r, c].sum():.2f}")]:
    ax.imshow(cost, cmap="Reds", vmin=0, vmax=1)
    viz.annotate_grid(ax, cost, fmt="{:.2f}", fontsize=11)
    for qi, gj in assign:
        ax.add_patch(Rectangle((gj - 0.5, qi - 0.5), 1, 1, fill=False, ec=BLUE, lw=4))
    ax.set_xticks([0, 1]); ax.set_xticklabels(["GT A", "GT B"]); ax.set_yticks([0, 1, 2]); ax.set_yticklabels(["query 1", "query 2", "query 3"])
    ax.set_title(title, fontsize=10)
ax = axes[2]; ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis("off")
q = [(1.0, 4.5), (1.0, 3.0), (1.0, 1.5)]; g = [(8.5, 4.0), (8.5, 2.0)]
for k, (x, y) in enumerate(q):
    ax.add_patch(FancyBboxPatch((x - 0.6, y - 0.4), 1.2, 0.8, boxstyle="round,pad=0.05", fc="#fdd0a2", ec="k")); ax.text(x, y, f"query {k + 1}", ha="center", va="center", fontsize=9)
for k, (x, y) in enumerate(g):
    ax.add_patch(FancyBboxPatch((x - 0.6, y - 0.4), 1.2, 0.8, boxstyle="round,pad=0.05", fc="#c7e9c0", ec="k")); ax.text(x, y, f"GT {'AB'[k]}", ha="center", va="center", fontsize=9)
for qi, gj in zip(r, c):
    ax.annotate("", (g[gj][0] - 0.65, g[gj][1]), (q[qi][0] + 0.65, q[qi][1]), arrowprops=dict(arrowstyle="-|>", lw=2.5, color=BLUE))
    ax.text(q[qi][0] + 0.25 * (g[gj][0] - q[qi][0]) + 0.6, q[qi][1] + 0.25 * (g[gj][1] - q[qi][1]) + 0.15, f"cost {cost[qi, gj]:.2f}", color=BLUE, ha="center")
unmatched = []                                                 # queries that got no GT
for k in range(3):
    if k not in r:
        unmatched.append(k)
for k in unmatched:
    ax.text(q[k][0] + 1.0, q[k][1], "-> learns 'no object'", fontsize=9, va="center", color=GREY)
ax.set_title("one-to-one: every GT gets exactly one query", fontsize=10)
plt.tight_layout(); plt.show()

What to look at: greedy lets GT A take query 1, its cheapest (0.20), which leaves GT B only the
expensive query 3 (0.85), total 1.05. The optimum gives query 1 to B and query 2 to A: each
choice is slightly worse for one object, but the sum is 0.55.

### Lab · Hungarian matching
**Predict first:** two queries sit almost exactly on the same object. After matching, how many of
them are trained to predict that object, and what is the other one trained to predict?
Drag the boxes to check, then work through the "Try this" list under the lab (6 minutes).

In [ ]:
labs.show("hungarian-matching")

### Exercise 2 · the DETR matching cost

`detr_cost(pred_logits (Q, C+1), pred_boxes (Q, 4) cxcywh in [0, 1], gt_labels (G,), gt_boxes (G, 4) cxcywh)`
-> `(Q, G)`. Use `softmax` for the class probabilities, `torch.cdist(..., p=1)` for the L1 term and
`torchvision.ops.generalized_box_iou` (needs xyxy: use `day2.box_cxcywh_to_xyxy_t`). This is the
matrix of the lab: rows are queries, columns are ground-truth objects.

<details class="api-ref" data-api="torch.cdist,torchvision.ops.generalized_box_iou"><summary><b>Functions you will need</b> (click to open): <code>torch.cdist</code> · <code>torchvision.ops.generalized_box_iou</code></summary>
<p><b><code>torch.cdist(a, b, p=2.0)</code></b><br>Distances between every row of a (N, D) and every row of b (M, D), as an (N, M) tensor. p=2 is the Euclidean distance, p=1 the L1 distance (sum of absolute differences): DETR&#x27;s box cost between Q predicted and G ground-truth boxes of 4 numbers each.</p><pre>&gt;&gt;&gt; a = torch.tensor([[0., 0.], [1., 1.]])
&gt;&gt;&gt; b = torch.tensor([[1., 0.], [3., 4.], [0., 0.]])
&gt;&gt;&gt; torch.cdist(a, b)                  # Euclidean, (2, 3)
tensor([[1.0000, 5.0000, 0.0000],
        [1.0000, 3.6056, 1.4142]])
&gt;&gt;&gt; torch.cdist(a, b, p=1)             # L1: |dx| + |dy|
tensor([[1., 7., 0.],
        [1., 5., 2.]])</pre><p>See also: <code>torchvision.ops.generalized_box_iou</code></p><p><b><code>torchvision.ops.generalized_box_iou(boxes1, boxes2)</code></b><br>Pairwise GIoU, like box_iou: every box of boxes1 (N, 4) against every box of boxes2 (M, 4), an (N, M) tensor, xyxy format. GIoU = IoU - (empty part of the smallest enclosing box) / (area of that box), in [-1, 1]: unlike IoU it keeps decreasing as boxes that do not overlap move apart.</p><pre>&gt;&gt;&gt; a = torch.tensor([[0., 0., 10., 10.]])
&gt;&gt;&gt; b = torch.tensor([[0., 0., 10., 10.], [20., 0., 30., 10.]])    # the same box; a disjoint one
&gt;&gt;&gt; torchvision.ops.generalized_box_iou(a, b)                       # right: 0 - (300 - 200) / 300
tensor([[ 1.0000, -0.3333]])</pre><p><i>Watch out:</i> It needs xyxy, while DETR predicts cxcywh: convert first (day2.box_cxcywh_to_xyxy_t or torchvision.ops.box_convert). For aligned pairs take .diag() of the result.</p><p>See also: <code>torchvision.ops.box_iou</code>, <code>torchvision.ops.generalized_box_iou_loss</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="Tensor.softmax"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>Tensor.softmax</code></summary>
<ul><li><code>logits.softmax(dim)</code> &nbsp; Turns scores (logits) into probabilities along dimension dim: exp(x) / sum(exp(x)). Every value ends up between 0 and 1 and they sum to 1 along dim; the order of the scores is kept, so the argmax does not change.</li></ul></details>

In [ ]:
from torchvision.ops import generalized_box_iou

def detr_cost(pred_logits, pred_boxes, gt_labels, gt_boxes, w_class=1.0, w_l1=5.0, w_giou=2.0):
    '''pred_logits (Q, C+1), pred_boxes (Q, 4) cxcywh in [0, 1], gt_labels (G,), gt_boxes (G, 4) cxcywh
    -> the matching cost (Q, G): entry [i, j] says how badly query i fits ground-truth object j.'''
    # 1. the class probabilities of every query
    probs = pred_logits.softmax(-1)                                       # (Q, C+1)
    # 2. class cost: minus the probability each query gives to the class of each object
    c_class = -probs[:, gt_labels]                                        # (Q, G)
    # 3. L1 distance between every predicted box and every GT box
    c_l1 = torch.cdist(pred_boxes, gt_boxes, p=1)                         # (Q, G)
    # 4. GIoU needs corner coordinates
    pred_xyxy = day2.box_cxcywh_to_xyxy_t(pred_boxes)                     # (Q, 4)
    gt_xyxy = day2.box_cxcywh_to_xyxy_t(gt_boxes)                         # (G, 4)
    # 5. GIoU cost: a high GIoU is a low cost
    c_giou = -generalized_box_iou(pred_xyxy, gt_xyxy)                     # (Q, G)
    # 6. the weighted sum
    return w_class * c_class + w_l1 * c_l1 + w_giou * c_giou

tests.check_detr_cost(detr_cost)

### Watch set prediction happen

No network yet - just **6 free "queries"**, each a learnable box and a learnable class
distribution over `circle / square / triangle / no-object`. Every step we match them to the 3
objects of a shapes image with your cost + Hungarian matching and apply DETR's loss: cross-entropy
for all queries (unmatched ones -> "no object", down-weighted 0.1), L1 + GIoU for the matched ones.
The queries *compete*; the losers drift into "no object", like the dashed queries in the lab.
That is why DETR needs no NMS.

<details class="api-ref" data-api="nn.Parameter,torch.full"><summary><b>Functions used here</b> (click to open): <code>nn.Parameter</code> · <code>torch.full</code></summary>
<p><b><code>nn.Parameter(tensor)</code></b><br>Marks a tensor as a learnable parameter: requires_grad is switched on, and when it is assigned as an attribute of an nn.Module it is registered, so model.parameters() (and thus the optimizer) includes it. DETR&#x27;s learned object queries are such parameters; D2-04 also hands two of them straight to an optimizer.</p><pre>&gt;&gt;&gt; q = nn.Parameter(torch.zeros(6, 4))
&gt;&gt;&gt; q.requires_grad
True
&gt;&gt;&gt; class Queries(nn.Module):
...     def __init__(self):
...         super().__init__()
...         self.q = nn.Parameter(torch.zeros(6, 4))
&gt;&gt;&gt; [(name, tuple(p.shape)) for name, p in Queries().named_parameters()]
[(&#x27;q&#x27;, (6, 4))]</pre><p>See also: <code>Module.parameters</code>, <code>Tensor.requires_grad_</code></p><p><b><code>torch.full(size, fill_value)</code></b><br>A new tensor of the given size with every element equal to fill_value. The dtype follows the value: an int gives int64 (what F.cross_entropy wants as class targets), a float gives float32. D2-04 starts every query&#x27;s target as &quot;no object&quot; this way and then overwrites the matched ones.</p><pre>&gt;&gt;&gt; t = torch.full((5,), 3)                          # 5 queries, class 3 = &quot;no object&quot;
&gt;&gt;&gt; t[torch.tensor([0, 2])] = torch.tensor([1, 0])   # two matched queries get real classes
&gt;&gt;&gt; t, t.dtype
(tensor([1, 3, 0, 3, 3]), torch.int64)
&gt;&gt;&gt; torch.full((2, 2), 0.5)
tensor([[0.5000, 0.5000],
        [0.5000, 0.5000]])</pre><p>See also: <code>torch.zeros</code>, <code>torch.full_like</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="torch.tensor,torch.randn,torch.zeros,torch.sigmoid,torch.optim.Adam,F.cross_entropy,Optimizer.zero_grad,Tensor.backward,Optimizer.step,torch.no_grad"><summary><b>Functions from earlier notebooks, used again here: the training loop</b> (click to open): <code>torch.tensor</code> · <code>torch.randn</code> · <code>torch.zeros</code> · <code>torch.sigmoid</code> · <code>torch.optim.Adam</code> · <code>F.cross_entropy</code> · <code>Optimizer.zero_grad</code> · <code>Tensor.backward</code> · <code>Optimizer.step</code> · <code>torch.no_grad</code></summary>
<ul><li><code>torch.tensor(data, dtype=None, device=None)</code> &nbsp; Builds a new tensor from a Python number, a (nested) list or a NumPy array, always copying the data. The dtype is inferred: integers give int64, floats give float32; pass dtype= to choose.</li><li><code>torch.randn(*size, generator=None)</code> &nbsp; A tensor of the given shape filled with random numbers from the standard normal distribution (mean 0, standard deviation 1); 0.4 * torch.randn(n) has standard deviation 0.4. The course uses it for noise and for random test inputs.</li><li><code>torch.zeros(*size, dtype=None, device=None)</code> &nbsp; A new tensor of the given shape filled with 0 (float32 unless dtype says otherwise). Used to allocate outputs and to build test inputs of a given shape.</li><li><code>torch.sigmoid(x)</code> &nbsp; Squashes every value (a logit) into (0, 1): sigmoid(x) = 1 / (1 + exp(-x)). Turns an objectness logit into a probability; D2-03 and D2-04 also use it to keep predicted box coordinates between 0 and 1.</li><li><code>torch.optim.Adam(params, lr=0.001)</code> &nbsp; The Adam optimizer. Like SGD it follows the gradient, but it divides each parameter&#x27;s step by a running estimate of that gradient&#x27;s size, so a step moves about lr whatever the scale of the gradient. D2-01 uses it to move four box coordinates with the same step length for every loss; D2-04 to train free queries.</li><li><code>F.cross_entropy(logits, target, label_smoothing=0.0)</code> &nbsp; The classification loss: logits (B, C) are raw scores, target is either the class indices (B,) as int64 or a (B, C) float tensor of probabilities (soft targets, e.g. from Mixup or CutMix). It applies log_softmax itself and returns the mean of -log p(correct class) over the batch. label_smoothing=eps trains against (1 - eps) on the true class plus eps / C spread over all classes.</li><li><code>optimizer.zero_grad()</code> &nbsp; Clears the .grad of every parameter the optimiser updates (sets it to None), so that the next backward() starts from zero. Call it in every training step, before loss.backward().</li><li><code>loss.backward()</code> &nbsp; Computes the gradient of a single number (usually the loss) with respect to every tensor it was computed from that has requires_grad=True, and adds it to that tensor&#x27;s .grad. One call fills the gradients of all the parameters of a model.</li><li><code>optimizer.step()</code> &nbsp; Updates every parameter the optimiser was given, using the gradients now in .grad; for plain SGD, w &lt;- w - lr * w.grad. Call it after loss.backward().</li><li><code>with torch.no_grad(): ...</code> &nbsp; A with block in which autograd records nothing: results have requires_grad=False, and computing them takes less memory and time. Use it when you only evaluate a model (testing, plotting its predictions), never around the training step itself.</li></ul></details>

<details class="api-ref" data-api="np.random.default_rng,Tensor.abs,Tensor.mean,Tensor.diag,Tensor.argmax,Tensor.max,Tensor.tolist"><summary><b>Functions from earlier notebooks, used again here: loss terms and read-out</b> (click to open): <code>np.random.default_rng</code> · <code>Tensor.abs</code> · <code>Tensor.mean</code> · <code>Tensor.diag</code> · <code>Tensor.argmax</code> · <code>Tensor.max</code> · <code>Tensor.tolist</code></summary>
<ul><li><code>rng = np.random.default_rng(seed)</code> &nbsp; Creates a NumPy random number generator; the same seed gives the same numbers on every run. Draw from it with rng.normal(size=...) (normal distribution), rng.random(size) (uniform in [0, 1)), rng.integers(low, high, size) or rng.permutation(n).</li><li><code>x.abs()</code> &nbsp; The absolute value of every element. (a - b).abs().max() is the largest difference between two tensors: the usual check that two results agree.</li><li><code>x.mean(dim=None, keepdim=False)</code> &nbsp; The average of all elements, or along dim (a number or a tuple), which then disappears from the shape. For a batch (B, C, H, W), mean(dim=(2, 3)) averages over the pixels and gives (B, C): global average pooling. NumPy arrays have the same method, with axis= for dim=.</li><li><code>x.diag()</code> &nbsp; The tensor version of np.diag: the diagonal of a 2-D tensor as a 1-D tensor (or a diagonal matrix from a 1-D tensor). 1 - box_iou(p, g).diag() is the IoU loss of the aligned pairs p[i], g[i].</li><li><code>x.argmax(dim)</code> &nbsp; The position of the largest value along dim. For logits (B, num_classes), x.argmax(dim=1) is the predicted class of every sample, shape (B,). Without dim it gives the position in the flattened tensor.</li><li><code>x.max() / x.max(dim)</code> &nbsp; x.max() is the largest element (a 0-dimensional tensor); x.max(dim) works along one dimension and returns two tensors, the maximum values and their positions (indices). NumPy arrays have the same method: arr.max() and arr.max(axis=...), which returns only the values.</li><li><code>x.tolist()</code> &nbsp; Converts a tensor into (nested) Python lists of plain Python numbers, for printing a few values readably. A 0-dimensional tensor gives a single number, like .item().</li></ul></details>

In [ ]:
cost_fn = detr_cost if tests.check_detr_cost(detr_cost, verbose=False) else day2.detr_cost
match_fn = hungarian_match if tests.check_hungarian(hungarian_match, verbose=False) else day2.hungarian_match
s_img, s_boxes, s_labels, _ = shapes.make_sample(np.random.default_rng(5), 128, max_objects=3, min_r=14, max_r=26)
gt_b = torch.tensor(boxes.xyxy_to_cxcywh(s_boxes) / 128, dtype=torch.float32); gt_l = torch.tensor(s_labels)
torch.manual_seed(3)
Q, NOOBJ = 6, 3
q_box = torch.nn.Parameter(torch.randn(Q, 4) * 0.8)                       # sigmoid -> cxcywh in (0, 1)
q_cls = torch.nn.Parameter(torch.zeros(Q, 4))
opt = torch.optim.Adam([q_box, q_cls], lr=0.05)
weight = torch.tensor([1.0, 1.0, 1.0, 0.1])                              # "no object" down-weighted
snaps = {}                                                               # step -> (boxes, class probabilities), for the figure
hist_match = []                                                          # after every step: {GT index: query index}
for step in range(301):
    pb = torch.sigmoid(q_box)                                            # (Q, 4) the query boxes, cxcywh in (0, 1)
    if step in (0, 15, 40, 300):                                         # snapshot before the update
        snaps[step] = (pb.detach().numpy().copy(), q_cls.softmax(-1).detach().numpy().copy())
    # 1. match: the cost matrix, then the Hungarian assignment (no gradient needed)
    with torch.no_grad():
        cost_matrix = cost_fn(q_cls, pb, gt_l, gt_b)                     # (Q, G)
        qi, gj = match_fn(cost_matrix)                                   # query qi[k] <-> object gj[k]
    # 2. class loss for every query: target "no object", except the GT class for the matched queries
    tgt = torch.full((Q,), NOOBJ)
    tgt[qi] = gt_l[gj]
    l_cls = F.cross_entropy(q_cls, tgt, weight=weight)
    # 3. box losses for the matched pairs only: L1 and GIoU
    matched_pred = pb[qi]                                                # (G, 4)
    matched_gt = gt_b[gj]                                                # (G, 4)
    l1_per_pair = (matched_pred - matched_gt).abs().sum(-1)              # (G,)
    l_l1 = l1_per_pair.mean()
    pred_xyxy = day2.box_cxcywh_to_xyxy_t(matched_pred)
    gt_xyxy = day2.box_cxcywh_to_xyxy_t(matched_gt)
    giou_matrix = torchvision.ops.generalized_box_iou(pred_xyxy, gt_xyxy)   # (G, G): pair k sits on the diagonal
    l_giou = (1 - giou_matrix.diag()).mean()
    # 4. DETR's loss, one optimiser step
    loss = l_cls + 5 * l_l1 + 2 * l_giou
    opt.zero_grad()
    loss.backward()
    opt.step()
    # 5. remember the assignment of this step
    assignment = {}
    for g_index, q_index in zip(gj.tolist(), qi.tolist()):
        assignment[g_index] = q_index
    hist_match.append(assignment)
fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
for ax, (step, (pb, prob)) in zip(axes, snaps.items()):
    ax.imshow(s_img); ax.axis("off"); ax.set_xlim(-10, 138); ax.set_ylim(138, -10)
    xyxy = boxes.cxcywh_to_xyxy(pb * 128)
    for k in range(Q):
        lab_k = prob[k].argmax(); obj = lab_k != NOOBJ
        c = viz.color(k)
        ax.add_patch(Rectangle(xyxy[k, :2], *(xyxy[k, 2:] - xyxy[k, :2]), fill=False, ec=c, lw=2.2 if obj else 1, ls="-" if obj else ":"))
        if obj:
            ax.text(xyxy[k, 0] + 1, xyxy[k, 1] + 8, f"q{k}: {shapes.CLASS_NAMES[lab_k]} {prob[k].max():.2f}",
                    fontsize=7, color="white", bbox=dict(fc=c, ec="none", pad=0.5, alpha=0.85))
    n_obj = int((prob.argmax(1) != NOOBJ).sum())
    ax.set_title(f"step {step}: {n_obj} queries predict an object, {Q - n_obj} 'no object'", fontsize=10)
fig.suptitle("6 learnable queries, 3 objects: matched queries converge, the others learn 'no object' (dotted)", y=1.02)
plt.tight_layout(); plt.show()
switches = 0                                                             # steps whose assignment differs from the step before
for t in range(1, len(hist_match)):
    if hist_match[t] != hist_match[t - 1]:
        switches += 1
print(f"the query-object assignment changed {switches} times in 300 steps. In a real DETR the queries are produced by a "
      "network from changing images, and early on the assignment flips constantly - one reason DETR trains slowly.")

**DETR's weak spots and their fixes.** The original needs ~500 epochs on COCO (vs ~12-36 for Faster
R-CNN) because the matching is unstable early on and global attention over all pixels is slow to
learn locality; it is also weak on small objects (single-scale, stride 32). **Deformable DETR** (2021)
attends to a few learned sampling points on multi-scale features (10x faster convergence);
**DN-DETR / DINO** (2022) add denoising queries; **RT-DETR** (2023) uses an efficient hybrid encoder and
IoU-aware query selection to run in real time and is now a standard choice alongside YOLO.

## 3 · Open-vocabulary detection

Every detector so far has a fixed list of classes baked into its last layer. **Open-vocabulary**
detectors (OWL-ViT 2022, Grounding DINO 2023, OWLv2, YOLO-World 2024) replace the class layer by a
**text encoder**: each candidate box gets an embedding, each prompt ("a cat's head", "a red
collar") gets an embedding, and the class score is their similarity - the same trick as CLIP
(Day 3). New classes cost nothing but a sentence. Great for prototyping and for auto-labelling
data that you then use to train a small fixed-class detector.

The demo below runs OWL-ViT through Hugging Face `transformers` if it is installed
(`pip install transformers`, ~600 MB of weights on first use); otherwise it draws the mechanism.

<details class="api-ref" data-api="transformers.pipeline"><summary><b>Functions used here</b> (click to open): <code>transformers.pipeline</code></summary>
<p><b><code>pipeline(task, model=&quot;org/name&quot;)</code></b><br>Hugging Face&#x27;s one-line inference API (pip install transformers): returns a ready-to-call object that downloads the named model from the Hugging Face Hub on first use (cached afterwards) and does its own pre- and post-processing. For task=&quot;zero-shot-object-detection&quot;, ov(pil_image, candidate_labels=[...], threshold=0.1) returns a list of dicts {&quot;score&quot;, &quot;label&quot;, &quot;box&quot;: {&quot;xmin&quot;, &quot;ymin&quot;, &quot;xmax&quot;, &quot;ymax&quot;}} in pixels.</p><p><i>Watch out:</i> The first call downloads hundreds of MB and needs internet access: wrap it in try / except, as D2-04 does.</p><p>See also: <code>torchvision.models.detection.fcos_resnet50_fpn</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="torchkit.get_pet_datasets"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torchkit.get_pet_datasets</code></summary>
<ul><li><code>train, val, classes = torchkit.get_pet_datasets(size=224, target=&quot;category&quot;, subset=None)</code> &nbsp; Course helper. Returns (train, val, classes) for Oxford-IIIT Pet (run tools/download_data.py first); images are float (3, size, size), resized and normalised with the ImageNet mean and std. target chooses the label: &quot;category&quot; gives (image, breed index 0..36) and the 37 breed names; &quot;segmentation&quot; gives (image, int64 mask (H, W)) with 0 background, 1 pet, 2 border; &quot;detection&quot; gives (image, {&quot;boxes&quot;: (N, 4) as x1, y1, x2, y2 in pixels, &quot;labels&quot;: (N,)}) for the pet&#x27;s head. subset=N keeps a class-balanced N training (and N // 4 validation) images for quick experiments.</li></ul></details>

In [ ]:
from PIL import Image
_, pet_val, _ = torchkit.get_pet_datasets(target="detection")
ov_img = Image.open(pet_val.image_path(40)).convert("RGB")
prompts = ["a dog's head", "a dog's ear", "a dog's nose", "grass"]
try:
    from transformers import pipeline
    ov = pipeline("zero-shot-object-detection", model="google/owlvit-base-patch32")
    t0 = time.time(); found = ov(ov_img, candidate_labels=prompts, threshold=0.1); dt = time.time() - t0
    fig, ax = plt.subplots(figsize=(7, 6)); ax.imshow(ov_img); ax.axis("off")
    for d in found:
        b = d["box"]; c = viz.color(prompts.index(d["label"]))
        ax.add_patch(Rectangle((b["xmin"], b["ymin"]), b["xmax"] - b["xmin"], b["ymax"] - b["ymin"], fill=False, ec=c, lw=2))
        ax.text(b["xmin"], b["ymin"] - 3, f"{d['label']} {d['score']:.2f}", color="white", fontsize=8, bbox=dict(fc=c, ec="none", pad=1))
    ax.set_title(f"OWL-ViT, prompts = {prompts}  ({dt:.1f} s)"); plt.show()
except Exception as e:  # ImportError, no internet, ...
    print(f"open-vocabulary demo skipped ({type(e).__name__}: {str(e)[:80]}). Showing the mechanism instead.")
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw=dict(width_ratios=[1, 1.25]))
    axes[0].imshow(ov_img); axes[0].axis("off"); axes[0].set_title("image -> ViT -> one box + one embedding per patch token")
    W_, H_ = ov_img.size
    cand = [(0.25, 0.08, 0.75, 0.55), (0.2, 0.05, 0.38, 0.32), (0.4, 0.3, 0.6, 0.48), (0.0, 0.7, 1.0, 1.0)]
    for k, (a, b_, c_, d_) in enumerate(cand):
        axes[0].add_patch(Rectangle((a * W_, b_ * H_), (c_ - a) * W_, (d_ - b_) * H_, fill=False, ec=viz.color(k), lw=2, ls="--"))
        axes[0].text(a * W_ + 3, b_ * H_ + 14, f"box {k}", color="white", fontsize=9, bbox=dict(fc=viz.color(k), ec="none", pad=1))
    ax = axes[1]; ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis("off")
    sim = np.array([[0.9, 0.2, 0.3, 0.0], [0.3, 0.8, 0.1, 0.0], [0.4, 0.1, 0.85, 0.0], [0.0, 0.0, 0.0, 0.7]])
    ax.imshow(sim, cmap="Greens", extent=(3, 9, 0.5, 4.5), vmin=0, vmax=1, aspect="auto")
    for i in range(4):
        ax.text(2.9, 4.0 - i, f"box {i}", ha="right", va="center", color=viz.color(i), weight="bold")
        ax.text(3.75 + 1.5 * i, 4.7, prompts[i], ha="center", fontsize=9, rotation=15)
    ax.text(6, 0.1, "illustrative similarity = cos(box embedding, text embedding) -> class score", ha="center", fontsize=9)
    ax.text(6, 5.6, "text encoder: one embedding per prompt", ha="center", fontsize=10, weight="bold")
    plt.tight_layout(); plt.show()

## 4 · Fine-tune a modern anchor-free detector and compare

We fine-tune torchvision's **FCOS** (anchor-free, focal loss, centre-ness - everything from D2-03)
on **exactly the same** Penn-Fudan subset, resolution and budget as the Faster R-CNN of D2-02, so the
comparison is fair: 40 training images at 320 px, 2 epochs (40 iterations of 2 images) capped at
3 minutes, evaluated with the same mAP code on the same 20 validation images. The provided checkpoints
in `data/checkpoints/` were produced by these exact cells without the time cap; delete them to train
yourself. The D2-02 model is re-trained here if its checkpoint is missing.

One difference: FCOS gets **LR 0.003** instead of 0.01. With 0.01 (fine for Faster R-CNN) the dense
focal-loss head kept oscillating and reached only mAP@0.5 = 0.72 in the same 40 iterations; with 0.003
it converges. Learning rate is the first knob to check when a fine-tune underperforms.

### Exercise 3 · adapt FCOS to 2 classes

FCOS has no separate predictor module: its `head.classification_head` ends in a 3 x 3 conv
`cls_logits` with `num_anchors * num_classes` outputs (num_anchors = 1, anchor-free). Replace
**only that conv** (keep the pretrained tower), set `classification_head.num_classes = 2`, and
initialise the new conv like RetinaNet (weights $\mathcal N(0, 0.01)$, bias $-\log\frac{1-0.01}{0.01}$).

<details class="api-ref" data-api="nn.init.normal_"><summary><b>Functions you will need</b> (click to open): <code>nn.init.normal_</code></summary>
<p><b><code>nn.init.normal_(tensor, mean=0.0, std=1.0)</code></b><br>Fills a parameter in place with random values from a normal distribution. A new classification layer in RetinaNet / FCOS gets weights with std=0.01 (so small that the output is set by the bias) and its bias from nn.init.constant_.</p><pre>&gt;&gt;&gt; conv = nn.Conv2d(256, 2, 3, padding=1)
&gt;&gt;&gt; _ = nn.init.normal_(conv.weight, std=0.01)
&gt;&gt;&gt; round(conv.weight.std().item(), 3)
0.01</pre><p>See also: <code>nn.init.constant_</code>, <code>nn.init.zeros_</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="torchvision.models.detection.fcos_resnet50_fpn,nn.Conv2d,nn.init.constant_"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torchvision.models.detection.fcos_resnet50_fpn</code> · <code>nn.Conv2d</code> · <code>nn.init.constant_</code></summary>
<ul><li><code>fcos_resnet50_fpn(weights=None, weights_backbone=None, num_classes=None, min_size=800, max_size=1333)</code> &nbsp; Builds torchvision&#x27;s FCOS: anchor-free (one prediction per feature-map location, so num_anchors = 1), ResNet-50 + FPN, focal loss and centre-ness. Same interface as the other torchvision detectors: a list of (3, H, W) images in [0, 1]; eval mode returns per image a dict with &quot;boxes&quot;, &quot;labels&quot;, &quot;scores&quot;, train mode with targets a dict of losses. Its class layer is model.head.classification_head.cls_logits.</li><li><code>nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, bias=True)</code> &nbsp; A convolution layer: out_channels learned kernels of size (in_channels, k, k), each slid over the input exactly like this notebook&#x27;s cross-correlation (no kernel flip). Input (B, C_in, H, W) -&gt; output (B, C_out, H_out, W_out), with H_out from the output-size formula. The weight has shape (C_out, C_in, k, k), the bias (C_out,).</li><li><code>nn.init.constant_(tensor, value)</code> &nbsp; Fills a parameter with one value, in place (the trailing _). Detectors set the bias of the last classification conv to -log((1 - pi) / pi) with pi = 0.01, so that at the start every anchor predicts &quot;1% object&quot; and the many background anchors do not blow up the first losses.</li></ul></details>

In [ ]:
import math
from torchvision.models.detection import fcos_resnet50_fpn
torch.manual_seed(0)
fcos = fcos_resnet50_fpn(weights="DEFAULT", min_size=320, max_size=512)
cls_head = fcos.head.classification_head           # the classification branch: tower + cls_logits conv
# 1. the head now has 2 classes
cls_head.num_classes = 2
# 2. the input channels of the old conv
in_channels = cls_head.cls_logits.in_channels      # 256
# 3. a new last conv with num_anchors * 2 outputs (num_anchors = 1: FCOS is anchor-free)
cls_head.cls_logits = nn.Conv2d(in_channels, cls_head.num_anchors * 2, 3, padding=1)
# 4. small random weights
nn.init.normal_(cls_head.cls_logits.weight, std=0.01)
# 5. the bias at the prior 0.01, as in RetinaNet
bias_value = -math.log((1 - 0.01) / 0.01)
nn.init.constant_(cls_head.cls_logits.bias, bias_value)

tests.check_box_predictor(fcos, num_classes=2)

<details class="api-ref" data-api="torchvision.models.detection.fasterrcnn_resnet50_fpn_v2,torchvision.models.detection.faster_rcnn.FastRCNNPredictor,Module.eval"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torchvision.models.detection.fasterrcnn_resnet50_fpn_v2</code> · <code>torchvision.models.detection.faster_rcnn.FastRCNNPredictor</code> · <code>Module.eval</code></summary>
<ul><li><code>fasterrcnn_resnet50_fpn_v2(weights=None, num_classes=None, min_size=800, max_size=1333, ...)</code> &nbsp; Builds torchvision&#x27;s Faster R-CNN (ResNet-50 + FPN, RPN, RoIAlign, box head; improved &quot;v2&quot; recipe, 43.7 M parameters). It takes a LIST of (3, H, W) float images in [0, 1] and resizes them itself (min_size, max_size). Train mode needs targets, a list of dicts with &quot;boxes&quot; (K, 4) xyxy pixels and &quot;labels&quot; (K,) int64 starting at 1 (0 means background), and returns a dict of the four losses; eval mode returns one dict per image with &quot;boxes&quot;, &quot;labels&quot;, &quot;scores&quot;.</li><li><code>FastRCNNPredictor(in_channels, num_classes)</code> &nbsp; The last layer of Faster R-CNN&#x27;s box head: two linear layers on each RoI&#x27;s feature vector, cls_score (num_classes logits) and bbox_pred (4 * num_classes deltas, one box per class). Installing a new one as model.roi_heads.box_predictor adapts a pretrained detector to your classes; num_classes counts the background, in_channels is the length of the RoI feature vector (cls_score.in_features of the old one).</li><li><code>model.eval()</code> &nbsp; Puts the model and all its layers into evaluation mode: dropout is switched off and BatchNorm uses the running averages collected during training, so a prediction no longer depends on the rest of the batch. Returns the model, so get_model(...).eval() works in one line.</li></ul></details>

In [ ]:
pf_train, pf_val = day2.penn_fudan_splits(n_train=40, n_val=20, seed=0, max_side=512)

def train_fcos():
    # only called when there is no saved checkpoint yet
    return day2.train_detector(fcos, pf_train, epochs=2, batch_size=2, lr=0.003, max_minutes=3, log_every=5)

fcos_hist = day2.load_or_train(fcos, "d2_04_fcos_pennfudan.pt", train_fcos)
fcos.eval()

# the D2-02 Faster R-CNN (same data, same budget) - loaded from its checkpoint, or trained now
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
FAST = dict(min_size=320, max_size=512, rpn_pre_nms_top_n_test=500, rpn_post_nms_top_n_test=100,
            rpn_pre_nms_top_n_train=1000, rpn_post_nms_top_n_train=300, box_batch_size_per_image=64)
torch.manual_seed(0)
frcnn = fasterrcnn_resnet50_fpn_v2(weights="DEFAULT", **FAST)
frcnn.roi_heads.box_predictor = FastRCNNPredictor(1024, 2)

def train_frcnn():
    # only called when there is no saved checkpoint yet
    return day2.train_detector(frcnn, pf_train, epochs=2, batch_size=2, lr=0.01, max_minutes=3, log_every=5)

day2.load_or_train(frcnn, "d2_02_frcnn_pennfudan.pt", train_frcnn)
frcnn.eval();

### Exercise 4 · a latency benchmark

Write `benchmark(model, sizes, repeats=3)` returning the **median** milliseconds of one forward
pass (batch of 1, `torch.no_grad()`, one warm-up call first) for a random `3 x s x s` image at
each size, as a dict `{size: milliseconds}` (e.g. `{320: 241.0, 480: ...}`). torchvision detectors resize internally, so set `model.transform.min_size = (s,)` and
`model.transform.max_size = s` before timing each size. Use `time.perf_counter()`.

<details class="api-ref" data-api="np.median"><summary><b>Functions you will need</b> (click to open): <code>np.median</code></summary>
<p><b><code>np.median(a, axis=None)</code></b><br>The middle value of the sorted data (the mean of the two middle values for an even count). A few slow outliers, such as a run disturbed by another program, shift the mean a lot but the median hardly at all: the right summary for repeated timings.</p><pre>&gt;&gt;&gt; ms = np.array([241., 238., 950., 244.])       # one run was disturbed
&gt;&gt;&gt; float(np.median(ms)), float(np.mean(ms))
(242.5, 418.25)</pre><p>See also: <code>np.mean</code>, <code>np.percentile</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="torch.rand"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torch.rand</code></summary>
<ul><li><code>torch.rand(*size, generator=None)</code> &nbsp; A tensor of the given shape filled with random numbers drawn uniformly from [0, 1). Scale and shift it for another range: torch.rand(n) * 4 - 2 is uniform in [-2, 2).</li></ul></details>

In [ ]:
@torch.no_grad()                                              # no gradients inside this function: faster
def benchmark(model, sizes=(320, 480, 640), repeats=3):
    '''model: a torchvision detector -> {size: median milliseconds of one forward pass on a random s x s image}.'''
    model.eval()
    out = {}
    for s in sizes:
        # 1. make the detector work at this size (it resizes every input internally)
        model.transform.min_size = (s,)
        model.transform.max_size = s
        x = [torch.rand(3, s, s)]                             # a batch of one random image (detectors take a list)
        # 2. warm-up, not timed
        model(x)
        # 3. `repeats` timed calls
        times = []
        for _ in range(repeats):
            t0 = time.perf_counter()
            model(x)
            times.append(time.perf_counter() - t0)            # seconds
        # 4. the median, in milliseconds
        median_seconds = float(np.median(times))
        out[s] = 1000 * median_seconds
    return out

In [ ]:
sizes = (320, 480, 640)
lat = {"Faster R-CNN v2": benchmark(frcnn, sizes), "FCOS": benchmark(fcos, sizes)}
for m in (frcnn, fcos):
    m.transform.min_size, m.transform.max_size = (320,), 512           # back to the training resolution
fig, ax = plt.subplots(figsize=(8, 4))
for (name, d), c in zip(lat.items(), [BLUE, ORANGE]):
    ax.plot(list(d), list(d.values()), "o-", color=c, lw=2, label=name)
    for s, v in d.items(): ax.text(s, v * 1.04, f"{v:.0f}", ha="center", fontsize=8, color=c)
ax.plot(sizes, [lat["FCOS"][320] * (s / 320) ** 2 for s in sizes], ":", color=GREY, label="ideal ~ pixels (s^2) from FCOS@320")
ax.set_xlabel("input size (px)"); ax.set_ylabel("ms / image (CPU, batch 1)"); ax.legend(); ax.grid(alpha=.3)
ax.set_title("latency grows with the number of pixels")
plt.tight_layout(); plt.show()

What to look at: from 320 to 640 px the side doubles and the pixel count grows 4x, but the measured
latency usually grows by less (compare each curve with the dotted line): part of the work, such as
the RoI head, NMS and framework overhead, does not depend on the resolution.

### Exercise 5 · the comparison table

Fill `rows` with one dict per model: name, parameters (millions, `torchkit.count_parameters`),
mAP@0.5 and mAP@[.5:.95] on `pf_val` (`day2.evaluate_map(model, pf_val)` returns both from one
inference pass, plus the predictions), and the latency at 320 px from `lat`. Then print it.

<details class="api-ref" data-api="torchkit.count_parameters"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torchkit.count_parameters</code></summary>
<ul><li><code>torchkit.count_parameters(model, trainable_only=False)</code> &nbsp; Course helper. The total number of parameters of a model, sum(p.numel() for p in model.parameters()); trainable_only=True counts only those with requires_grad (useful once layers are frozen for fine-tuning).</li></ul></details>

In [ ]:
rows = []                                                     # one dict per model
# (name in the table, key in lat, the model)
models = [("Faster R-CNN v2 (two-stage)", "Faster R-CNN v2", frcnn),
          ("FCOS (one-stage, anchor-free)", "FCOS", fcos)]
for name, key, model in models:
    # 1. one inference pass over the validation images: mAP and predictions
    res = day2.evaluate_map(model, pf_val)
    # 2. one row of the table
    row = {}
    row["name"] = name
    row["params"] = torchkit.count_parameters(model) / 1e6
    row["map50"] = res["map50"]
    row["map"] = res["map"]
    row["ms"] = lat[key][320]
    row["preds"] = res["preds"]
    # 3. into the table
    rows.append(row)
print(f"{'model':32s}{'params':>9}{'mAP@0.5':>9}{'mAP@[.5:.95]':>14}{'ms @320':>9}")
for row in rows:
    print(f"{row['name']:32s}{row['params']:8.1f}M{row['map50']:9.3f}{row['map']:14.3f}{row['ms']:9.0f}")

**What to expect.** With the provided checkpoints both models land at mAP@0.5 = 0.99 and
mAP@[.5:.95] = 0.78-0.79 on our 20 validation images - a tie (20 images cannot separate them).
With a GPU, the full 120-image training split and 800 px, both reach ~0.97-0.99 / ~0.75-0.85.
FCOS has ~25% fewer parameters and no RoI stage, so it is usually faster at the same resolution
(check the latency column for your machine). The differences between modern detectors on an easy,
small dataset are smaller than the differences caused by **resolution, data, learning rate and
training length** - measure on *your* data before choosing.

### Fine-tuning on your own data: Pet heads (single class)

The same 10 lines adapt FCOS to the course's running dataset, the Pet *head* boxes, at 320 px. We
use 48 training images and 2 epochs (about 3 CPU minutes, capped at 2.5 here; a provided checkpoint
makes re-runs instant). COCO has no "head" class: the detector has to learn a new definition of
"object" from 48 examples. Expect mAP@0.5 around 0.6-0.7 on 16 validation images: the top-1 box
usually sits on the head, but often too large (whole face + ears) or too small. With a GPU, the
2 900 training images and ~10 epochs, the same code reaches mAP@0.5 above 0.95.

<details class="api-ref" data-api="torchvision.transforms.v2.Compose,torchvision.transforms.v2.ToImage,torchvision.transforms.v2.ToDtype,Tensor.permute,np.argsort"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torchvision.transforms.v2.Compose</code> · <code>torchvision.transforms.v2.ToImage</code> · <code>torchvision.transforms.v2.ToDtype</code> · <code>Tensor.permute</code> · <code>np.argsort</code></summary>
<ul><li><code>v2.Compose([transform1, transform2, ...])</code> &nbsp; Chains transforms into one: calling the result applies them in the listed order, each on the output of the previous one. A whole per-image pipeline (resize, augment, convert to float, normalise) becomes one callable that a Dataset applies to every sample.</li><li><code>v2.ToImage()</code> &nbsp; Transform that turns a PIL image or an (H, W, C) NumPy array into an image tensor of shape (C, H, W). Values and dtype are kept (uint8 stays 0..255), so it is almost always followed by v2.ToDtype(torch.float32, scale=True) to get floats in [0, 1].</li><li><code>v2.ToDtype(torch.float32, scale=True)</code> &nbsp; Converts the image to another dtype. With scale=True it also rescales the values to the range of the new type: uint8 0 .. 255 becomes float 0.0 .. 1.0. The step between the cheap uint8 transforms and Normalize.</li><li><code>x.permute(*dims)</code> &nbsp; Reorders the dimensions: the argument lists the old positions in their new order. No data is copied. Matplotlib and image files use (H, W, C); PyTorch layers use (C, H, W); permute converts.</li><li><code>np.argsort(a)</code> &nbsp; The indices that would sort an array in ascending order; [::-1] reverses them for descending. a[np.argsort(a)] is sorted, and indexing other arrays with the same order keeps them aligned (class names, colours).</li></ul></details>

In [ ]:
from torchvision.transforms import v2
to_float = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
pet_ds_train, pet_ds_val, _ = torchkit.get_pet_datasets(target="detection", subset=(48, 16), seed=0, train_tf=to_float, eval_tf=to_float)
heads_train = day2.materialise(pet_ds_train, range(len(pet_ds_train)), max_side=400)
heads_val = day2.materialise(pet_ds_val, range(len(pet_ds_val)), max_side=400)
torch.manual_seed(0)
fcos_heads = fcos_resnet50_fpn(weights="DEFAULT", min_size=320, max_size=400)
h = fcos_heads.head.classification_head                    # the same 2-class change as in Exercise 3
h.num_classes = 2
h.cls_logits = nn.Conv2d(256, 2, 3, padding=1)
nn.init.normal_(h.cls_logits.weight, std=0.01)
nn.init.constant_(h.cls_logits.bias, -math.log(99))       # -log((1 - 0.01) / 0.01)

def train_heads():
    # only called when there is no saved checkpoint yet
    return day2.train_detector(fcos_heads, heads_train, epochs=2, batch_size=2, lr=0.003, max_minutes=2.5, log_every=6)

day2.load_or_train(fcos_heads, "d2_04_fcos_petheads.pt", train_heads)
heads_res = day2.evaluate_map(fcos_heads, heads_val)
heads_preds, heads_gts = heads_res["preds"], heads_res["gts"]
print(f"FCOS on Pet heads (16 val images): mAP@0.5 = {heads_res['map50']:.3f}, mAP@[.5:.95] = {heads_res['map']:.3f}")
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, i in zip(axes.flat, range(12)):
    ax.imshow(heads_val[i % len(heads_val)][0].permute(1, 2, 0)); ax.axis("off")
    viz.draw_boxes(ax, heads_gts[i % len(heads_val)]["boxes"], edgecolor=GREEN, lw=3, alpha=0.5)
    p = heads_preds[i % len(heads_val)]; k = np.argsort(-p["scores"])[:1]
    viz.draw_boxes(ax, p["boxes"][k], scores=p["scores"][k], edgecolor=RED, lw=1.8)
fig.suptitle("FCOS fine-tuned on Pet heads: top-1 prediction (red) vs GT (green)", y=1.0)
plt.tight_layout(); plt.show()

## 5 · Deployment

* **Resolution** is the biggest latency knob (cost grows with the number of pixels, see the plot);
  it is also an accuracy knob for small objects. Choose it on the validation set, per use case.
* **Batching** amortises overhead on GPUs; on CPUs and edge devices latency at batch 1 is what matters.
* **Export**: `torch.onnx.export` produces a framework-neutral graph that ONNX Runtime, TensorRT
  (NVIDIA), OpenVINO (Intel) or CoreML can optimise (layer fusion, FP16 / INT8 kernels). Detection
  post-processing (NMS) is often the hardest part to export - one more argument for NMS-free models.
* **Quantisation**: INT8 weights and activations cut size by 4x and latency by 2-4x on supported
  hardware; post-training quantisation needs a few hundred calibration images, quantisation-aware
  training recovers the last accuracy points.
* **Monitoring**: log the score distribution, the number of detections per image and box sizes in
  production. A shift (new camera, night images, a new product) shows up there long before anyone
  labels new data - compare with the validation distribution below.

<details class="api-ref" data-api="torch.onnx.export,np.bincount"><summary><b>Functions used here</b> (click to open): <code>torch.onnx.export</code> · <code>np.bincount</code></summary>
<p><b><code>torch.onnx.export(model, args, f, input_names=None, output_names=None, dynamic_axes=None)</code></b><br>Converts a model to ONNX, a framework-neutral format that ONNX Runtime, TensorRT, OpenVINO or CoreML can load and optimise: it records the operations the model performs on the example input args (a tensor or a tuple of tensors) and writes graph and weights to f (a file name or a file-like object such as io.BytesIO). input_names / output_names name the graph&#x27;s inputs and outputs; dynamic_axes marks dimensions, such as the batch, that may change at run time.</p><p><i>Watch out:</i> Needs the onnx package (and onnxscript for the default exporter of recent PyTorch versions); call model.eval() first. Detection post-processing such as NMS is often the part that fails to export.</p><p>See also: <code>Module.eval</code></p><p><b><code>np.bincount(x)</code></b><br>Counts how often each non-negative integer occurs: result[k] = how many times k appears in x, for k = 0 ... max(x). D2-04 turns &quot;detections per image&quot; into a histogram with it.</p><pre>&gt;&gt;&gt; np.bincount([0, 2, 2, 3, 2])           # 0 once, 1 never, 2 three times, 3 once
array([1, 0, 3, 1])</pre><p>See also: <code>torch.bincount</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.concatenate"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.concatenate</code></summary>
<ul><li><code>np.concatenate(arrays, axis=0)</code> &nbsp; Joins arrays end to end along an EXISTING axis; all other dimensions must match. axis=0 appends rows; axis=1 puts columns side by side, e.g. top-left corners (N, 2) and bottom-right corners (N, 2) into xyxy boxes (N, 4). The AP code uses it to pad a curve with one value at each end.</li></ul></details>

In [ ]:
# ONNX export of the D2-03-style mini detector (small, so the demo is instant)
import io
mini = day2.MiniDetector().eval()
try:
    import onnx  # noqa: F401
    buf = io.BytesIO()
    torch.onnx.export(mini, torch.randn(1, 3, 128, 128), buf, input_names=["image"], output_names=["grid"],
                      dynamic_axes={"image": {0: "batch"}, "grid": {0: "batch"}})
    print(f"exported MiniDetector to ONNX: {len(buf.getvalue()) / 1e6:.2f} MB")
except Exception as e:
    print(f"ONNX export skipped ({type(e).__name__}: {str(e)[:100]}). `pip install onnx onnxscript` to try it.")

# Production-style monitoring: the distribution of detection scores on validation images
for row in rows:                                         # the FCOS row of the table of Exercise 5
    if row["name"].startswith("FCOS"):
        fcos_preds = row["preds"]                        # one dict (boxes, scores, labels) per validation image
        break
score_arrays = []                                        # the scores of every image
n_det = []                                               # detections with score > 0.5, per image
for p in fcos_preds:
    score_arrays.append(p["scores"])
    n_det.append(int((p["scores"] > 0.5).sum()))
all_scores = np.concatenate(score_arrays)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
axes[0].hist(all_scores, bins=40, color=ORANGE); axes[0].set_xlabel("detection score"); axes[0].set_ylabel("count")
axes[0].set_title("score histogram, FCOS on Penn-Fudan val (torchvision drops scores < 0.2)")
cnt = np.bincount(n_det); axes[1].bar(range(len(cnt)), cnt, color=BLUE, width=0.7); axes[1].set_xlabel("detections > 0.5 per image")
axes[1].set_ylabel("images"); axes[1].set_xticks(range(len(cnt)))
axes[1].set_title("detections per image")
plt.tight_layout(); plt.show()

What to look at: the scores form two groups, many weak detections just above the 0.2 cut-off and a
cluster of confident ones near 0.7-0.9, with few in between. A production monitor would watch the
share of scores in the middle and the detections-per-image histogram: both shift when the input
data changes.

## 6 · Which detector would you pick?

### Exercise 6 · three scenarios (written)

For each scenario, name a detector family (and a size / resolution), the main risk, and how you
would validate the choice. There is no single right answer - argue with the trade-offs of today.

1. **Factory line**: count screws on a conveyor belt, fixed camera, 60 fps, an industrial PC with a
   small NVIDIA GPU, 500 labelled images, product must ship commercially next quarter.
2. **Wildlife camera traps**: 2 million unlabelled images, 30 rare species, latency irrelevant
   (batch processing overnight), biologists can label ~2 000 boxes.
3. **Retail shelf audit app**: phone camera, detect ~1 000 different product SKUs that change monthly,
   a few examples per SKU.

In [ ]:
answers = {
    "factory line": "Small real-time one-stage model (RT-DETR-R18 for a permissive licence, or a "
                    "YOLO-n/s with a commercial licence), fine-tuned at the lowest resolution that "
                    "still resolves a screw, exported to TensorRT FP16/INT8. Risk: tiny, touching "
                    "objects -> check recall on crowded frames and NMS (or an NMS-free model). "
                    "Validate on a held-out week of production video, measure fps on the target PC.",
    "camera traps": "Accuracy over speed: a large two-stage or DETR-family model (or an open-vocabulary "
                    "detector to pre-label 'animal' boxes, then human verification = cheap labels). "
                    "Detect 'animal' class-agnostically, classify species with a separate classifier "
                    "on crops (rare classes!). Risk: long tail and empty frames -> report per-species "
                    "recall and AP, not one mAP. Validate on cameras not seen in training.",
    "shelf audit": "Classes change monthly -> do not bake 1 000 SKUs into a detector head. Use a "
                   "class-agnostic 'product' detector (fine-tuned, mobile-sized, quantised) + an "
                   "embedding / retrieval model to recognise the SKU from a few reference photos "
                   "(or an open-vocabulary detector for prototyping). Risk: look-alike packages, "
                   "drift with new packaging -> monitor scores, re-index embeddings without retraining.",
}
for scenario, answer in answers.items():
    print(f"{scenario}: {answer}\n")

## 7 · Summary

* The YOLO line went grid -> anchors -> multi-scale -> anchor-free decoupled heads -> NMS-free;
  today's real-time detectors (YOLO 8/10/11, RT-DETR) span 1-15 ms on a T4 at 37-55 COCO mAP.
* DETR casts detection as set prediction: N queries, Hungarian matching with a class + L1 + GIoU
  cost, a "no object" class, no anchors and no NMS. Deformable DETR, DINO and RT-DETR fixed its slow
  training and made it real-time.
* Open-vocabulary detectors score boxes against text embeddings: classes become prompts.
* Fine-tuning recipe: COCO-pretrained model, replace the class layer (prior-initialised bias),
  small LR, short schedule; compare models on the *same* data, resolution and budget.
* Deployment: resolution drives latency; export (ONNX / TensorRT), quantise, and monitor score
  distributions for drift.

### Self-check
1. Why does DETR need a "no object" class, and why is its loss weight reduced (0.1)?
2. With 100 queries and 3 objects, how many predictions receive a box loss in one image?
3. What does YOLOv10 change at training time to remove NMS at inference?
4. You double the input resolution of a detector. Roughly how does latency change? What might improve?
5. Name two reasons to prefer RT-DETR or torchvision models over Ultralytics YOLO in a commercial product.
6. When is an open-vocabulary detector the right production choice, and when is it only a labelling tool?

### Go deeper
* Carion et al., *End-to-End Object Detection with Transformers* (DETR, ECCV 2020).
* Zhu et al., *Deformable DETR* (ICLR 2021).
* Lv et al. / Zhao et al., *DETRs Beat YOLOs on Real-time Object Detection* (RT-DETR, CVPR 2024).
* Wang et al., *YOLOv10: Real-Time End-to-End Object Detection* (NeurIPS 2024).
* Liu et al., *Grounding DINO: Marrying DINO with Grounded Pre-Training for Open-Set Object Detection* (2023).
* Ultralytics documentation (docs.ultralytics.com) - training, export and licence notes for the YOLO family.